# Visão Computacional: fissuras em concreto (GS 2026.2, FIAP)

Notebook **executável do início ao fim** para o enunciado de Visão Computacional (Manutenção Urbana).
Faz: preparar dados, treinar (MobileNetV2), escolher limiar na validação, avaliar no teste,
analisar acertos e erros, testar **suas fotos reais** e medir **latência e FPS** em **seu vídeo**.

**Como usar:** menu *Ambiente de execução → Alterar tipo → GPU T4* (opcional, acelera o treino) e depois *Executar tudo*.
Nas células marcadas com 📷 e 🎥 o Colab vai pedir para você enviar arquivos.

**Convenção dos nomes das fotos (célula 📷):** comece com `com_` (tem fissura) ou `sem_` (não tem) e, se quiser,
acrescente condições depois de `__`. Exemplos: `com_parede01.jpg`, `com_calcada02__sombra.jpg`,
`sem_fachada03__poucaluz.jpg`, `com_muro04__obstaculo__textura.jpg`.
Qualquer palavra depois de `__` vira uma condição na tabela. Sugeridas: `sombra`, `poucaluz`, `obstaculo`, `textura`, `contraluz`, `longe` (fachada inteira, para mostrar o limite de escala).

**Aviso de triagem:** fissura na imagem não define gravidade nem segurança estrutural.

## 1. Ambiente

In [ ]:
import platform, subprocess, sys
from pathlib import Path
BASE = Path('/content/vc')            # espelha a estrutura do repositório: codigo/, imagens/, evidencias/
(BASE / 'codigo').mkdir(parents=True, exist_ok=True)
(BASE / 'evidencias').mkdir(exist_ok=True)
(BASE / 'fotos_reais').mkdir(exist_ok=True)
subprocess.run('apt-get -qq install -y libarchive-tools > /dev/null', shell=True)
import torch
print('Python', sys.version.split()[0], '| torch', torch.__version__, '| GPU:', torch.cuda.is_available())
print(subprocess.run("lscpu | grep 'Model name'", shell=True, capture_output=True, text=True).stdout.strip())

## 2. Código do projeto (cópia dos scripts de `04-visao-computacional/codigo/`)

In [ ]:
%%writefile /content/vc/codigo/config.py
"""Configuração do módulo de Visão Computacional (classificação de fissuras).

Base: Concrete Crack Images for Classification (Mendeley, 40 mil imagens 227x227).
Baixe e extraia para 04-visao-computacional/imagens/ com as pastas:
    imagens/Positive/  (com fissura)
    imagens/Negative/  (sem fissura)
"""
import os
from pathlib import Path

RAIZ = Path(__file__).resolve().parent.parent
IMAGENS = Path(os.getenv("VC_IMAGENS", RAIZ / "imagens"))
SAIDA = RAIZ / "evidencias"
MANIFESTO = SAIDA / "manifesto_split.csv"
MODELO = Path(os.getenv("VC_MODELO", RAIZ / "codigo" / "modelo_fissuras.pt"))

CLASSES = ["sem_fissura", "com_fissura"]   # índice 0 e 1
PASTAS = {"Negative": 0, "Positive": 1}

AMOSTRA_POR_CLASSE = int(os.getenv("VC_AMOSTRA", "2000"))
TAMANHO_BLOCO = 100            # arquivos consecutivos tratados como mesma origem
PROPORCOES = (0.7, 0.15, 0.15)  # treino, validação, teste
SEMENTE = 42

TAMANHO_IMG = 224
MEDIA = (0.485, 0.456, 0.406)
DESVIO = (0.229, 0.224, 0.225)
EPOCAS = int(os.getenv("VC_EPOCAS", "5"))
BATCH = 32
# 0,45: escolhido na VALIDAÇÃO (recall >= 0,99 com maior precisão); ver evidencias/limiar_escolhido.json
LIMIAR_CONFIANCA = float(os.getenv("VC_LIMIAR", "0.45"))

In [ ]:
%%writefile /content/vc/codigo/dados.py
import csv

import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

import config

normaliza = transforms.Normalize(config.MEDIA, config.DESVIO)
tf_base = transforms.Compose([transforms.Resize((config.TAMANHO_IMG, config.TAMANHO_IMG)), transforms.ToTensor(), normaliza])
tf_treino = transforms.Compose([
    transforms.Resize((config.TAMANHO_IMG, config.TAMANHO_IMG)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(), normaliza,
])


class Fissuras(Dataset):
    def __init__(self, conjunto):
        with open(config.MANIFESTO, encoding="utf-8") as f:
            self.itens = [l for l in csv.DictReader(f) if l["conjunto"] == conjunto]
        for it in self.itens:  # o manifesto guarda caminho relativo; aqui vira absoluto para abrir a imagem
            it["relativo"] = it["arquivo"]
            it["arquivo"] = str(config.IMAGENS / it["arquivo"])
        self.tf = tf_treino if conjunto == "treino" else tf_base

    def __len__(self):
        return len(self.itens)

    def __getitem__(self, i):
        it = self.itens[i]
        img = Image.open(it["arquivo"]).convert("RGB")
        return self.tf(img), torch.tensor(int(it["rotulo"]))

In [ ]:
%%writefile /content/vc/codigo/treinar.py
"""Etapa 2: transferência de aprendizado com MobileNetV2 pré-treinada (ImageNet).

Justificativa: arquitetura leve, boa para CPU e para inferência em vídeo.
Congela o extrator de características e treina só a cabeça de classificação.
"""
import json
import time

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import models

import config
from dados import Fissuras


def criar_modelo():
    m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    for p in m.features.parameters():
        p.requires_grad = False
    m.classifier[1] = nn.Linear(m.last_channel, 2)
    return m


def main():
    torch.manual_seed(config.SEMENTE)
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    modelo = criar_modelo().to(dispositivo)
    carga_tr = DataLoader(Fissuras("treino"), batch_size=config.BATCH, shuffle=True, num_workers=0)
    carga_va = DataLoader(Fissuras("validacao"), batch_size=config.BATCH, num_workers=0)
    otim = torch.optim.Adam(modelo.classifier.parameters(), lr=1e-3)
    perda = nn.CrossEntropyLoss()
    historico, melhor = [], 0.0
    for ep in range(1, config.EPOCAS + 1):
        t0 = time.time()
        modelo.train()
        soma = 0.0
        for x, y in carga_tr:
            x, y = x.to(dispositivo), y.to(dispositivo)
            otim.zero_grad()
            l = perda(modelo(x), y)
            l.backward()
            otim.step()
            soma += l.item() * len(y)
        modelo.eval()
        certos = total = 0
        with torch.no_grad():
            for x, y in carga_va:
                p = modelo(x.to(dispositivo)).argmax(1).cpu()
                certos += (p == y).sum().item()
                total += len(y)
        acc = certos / total
        historico.append({"epoca": ep, "perda_treino": soma / len(carga_tr.dataset), "acc_validacao": acc, "seg": round(time.time() - t0, 1)})
        print(historico[-1])
        if acc >= melhor:
            melhor = acc
            torch.save(modelo.state_dict(), config.MODELO)
    (config.SAIDA / "historico_treino.json").write_text(json.dumps(historico, indent=2), encoding="utf-8")
    print("Melhor acurácia de validação:", melhor, "| modelo salvo em", config.MODELO)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/vc/codigo/inferir.py
"""Etapa 4: função de inferência usada pela API do app (01-front-end/api).

predizer(imagem) devolve rótulo, confiança e se há necessidade de inspeção.
Aviso: fissura na imagem não define gravidade nem segurança estrutural; serve para triagem.
"""
from functools import lru_cache

import torch
from PIL import Image

import config
from dados import tf_base
from treinar import criar_modelo

AVISO = "Triagem apenas. A imagem não permite avaliar a gravidade nem a segurança da estrutura."


@lru_cache(maxsize=1)
def carregar_modelo():
    m = criar_modelo()
    m.load_state_dict(torch.load(config.MODELO, map_location="cpu"))
    return m.eval()


def predizer(imagem: Image.Image, limiar: float = config.LIMIAR_CONFIANCA) -> dict:
    x = tf_base(imagem.convert("RGB")).unsqueeze(0)
    with torch.no_grad():
        prob = torch.softmax(carregar_modelo()(x), 1)[0]
    p_fissura = float(prob[1])
    return {
        "rotulo": config.CLASSES[1] if p_fissura >= limiar else config.CLASSES[0],
        # confiança = probabilidade da classe devolvida (coerente com o rótulo, mesmo com limiar != 0,5)
        "confianca": round(p_fissura if p_fissura >= limiar else 1 - p_fissura, 4),
        "probabilidade_fissura": round(p_fissura, 4),
        "inspecao_recomendada": p_fissura >= limiar,
        "aviso": AVISO,
    }


if __name__ == "__main__":
    import sys
    print(predizer(Image.open(sys.argv[1])))

In [ ]:
%%writefile /content/vc/codigo/preparar_dados.py
"""Etapa 1: amostra os dados e separa treino/validação/teste sem vazamento.

O dataset foi gerado recortando ~458 imagens grandes e não informa a origem de cada recorte.
Como arquivos com numeração próxima tendem a vir da mesma imagem original, a separação é
feita por BLOCOS de arquivos consecutivos: um bloco inteiro vai para um único conjunto.
Limitação: é uma aproximação, não uma garantia. Declarar isso no relatório.
"""
import csv
import random

import config


def listar(pasta):
    return sorted(p for p in pasta.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})


def main():
    rnd = random.Random(config.SEMENTE)
    linhas = []
    for nome_pasta, rotulo in config.PASTAS.items():
        pasta = config.IMAGENS / nome_pasta
        if not pasta.exists():
            raise SystemExit(f"Pasta não encontrada: {pasta}. Baixe o dataset (ver config.py).")
        arquivos = listar(pasta)
        blocos = [arquivos[i:i + config.TAMANHO_BLOCO] for i in range(0, len(arquivos), config.TAMANHO_BLOCO)]
        rnd.shuffle(blocos)
        escolhidos, total = [], 0
        for b in blocos:  # amostra por blocos inteiros até atingir a quantidade
            if total >= config.AMOSTRA_POR_CLASSE:
                break
            escolhidos.append(b)
            total += len(b)
        n = len(escolhidos)
        corte1 = round(n * config.PROPORCOES[0])
        corte2 = corte1 + round(n * config.PROPORCOES[1])
        for i, bloco in enumerate(escolhidos):
            conjunto = "treino" if i < corte1 else "validacao" if i < corte2 else "teste"
            for arq in bloco:
                # caminho relativo a config.IMAGENS (portável entre máquinas)
                linhas.append({"arquivo": arq.relative_to(config.IMAGENS).as_posix(), "rotulo": rotulo, "conjunto": conjunto, "bloco": f"{nome_pasta}-{i}"})
    config.SAIDA.mkdir(parents=True, exist_ok=True)
    with open(config.MANIFESTO, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["arquivo", "rotulo", "conjunto", "bloco"])
        w.writeheader()
        w.writerows(linhas)
    for c in ("treino", "validacao", "teste"):
        print(c, sum(1 for l in linhas if l["conjunto"] == c))
    # checagem de vazamento: nenhum bloco em mais de um conjunto
    por_bloco = {}
    for l in linhas:
        por_bloco.setdefault(l["bloco"], set()).add(l["conjunto"])
    assert all(len(v) == 1 for v in por_bloco.values()), "vazamento entre conjuntos"
    print("Manifesto salvo em", config.MANIFESTO)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/vc/codigo/escolher_limiar.py
"""Escolhe o limiar de decisão na VALIDAÇÃO e só então mede no TESTE (sem contaminar o teste).

Critério definido antes de olhar os números, por ser triagem (perder fissura custa mais que
inspeção desnecessária): entre os limiares com recall >= 0,99 na validação, o de maior precisão
(empate: o maior limiar). Se nenhum atingir, usa o de maior recall.
Saída: evidencias/limiar_validacao.csv, evidencias/limiar_escolhido.json
"""
import csv
import json

import torch
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader

import config
from dados import Fissuras
from treinar import criar_modelo

RECALL_ALVO = 0.99
LIMIARES = [round(0.05 * k, 2) for k in range(2, 19)]  # 0,10 a 0,90


def probabilidades(modelo, conjunto):
    y, p = [], []
    with torch.no_grad():
        for x, r in DataLoader(Fissuras(conjunto), batch_size=config.BATCH):
            p += torch.softmax(modelo(x), 1)[:, 1].tolist()
            y += r.tolist()
    return y, p


def metricas(y, p, lim):
    pred = [int(v >= lim) for v in p]
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return {"limiar": lim, "precisao": round(precision_score(y, pred), 4), "recall": round(recall_score(y, pred), 4),
            "f1": round(f1_score(y, pred), 4), "fn": int(fn), "fp": int(fp)}


def main():
    modelo = criar_modelo()
    modelo.load_state_dict(torch.load(config.MODELO, map_location="cpu"))
    modelo.eval()
    yv, pv = probabilidades(modelo, "validacao")
    tabela = [metricas(yv, pv, l) for l in LIMIARES]
    with open(config.SAIDA / "limiar_validacao.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(tabela[0]))
        w.writeheader()
        w.writerows(tabela)
    for t in tabela:
        print("validacao", t)
    ok = [t for t in tabela if t["recall"] >= RECALL_ALVO]
    escolhido = max(ok, key=lambda t: (t["precisao"], t["limiar"])) if ok else max(tabela, key=lambda t: (t["recall"], t["limiar"]))
    yt, pt = probabilidades(modelo, "teste")
    no_teste = metricas(yt, pt, escolhido["limiar"])
    padrao_teste = metricas(yt, pt, 0.5)
    res = {"criterio": f"recall >= {RECALL_ALVO} na validação, maior precisão", "limiar_escolhido": escolhido["limiar"],
           "validacao": escolhido, "teste_no_limiar_escolhido": no_teste, "teste_no_limiar_0.5": padrao_teste}
    (config.SAIDA / "limiar_escolhido.json").write_text(json.dumps(res, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(res, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/vc/codigo/avaliar.py
"""Etapa 3: avalia no conjunto de TESTE (nunca visto no treino ou na validação).

Gera precisão, recall, F1, matriz de confusão e lista de erros para análise
(sombra, textura, iluminação, obstáculos).
"""
import csv

import matplotlib
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from torch.utils.data import DataLoader

import config
from dados import Fissuras
from treinar import criar_modelo

matplotlib.use("Agg")


def main():
    modelo = criar_modelo()
    modelo.load_state_dict(torch.load(config.MODELO, map_location="cpu"))
    modelo.eval()
    ds = Fissuras("teste")
    y_real, y_prev, confs = [], [], []
    with torch.no_grad():
        for x, y in DataLoader(ds, batch_size=config.BATCH):
            prob = torch.softmax(modelo(x), 1)
            p_fissura = prob[:, 1]
            pred = (p_fissura >= config.LIMIAR_CONFIANCA).long()  # mesmo limiar usado na API
            conf = torch.where(pred == 1, p_fissura, 1 - p_fissura)
            y_real += y.tolist()
            y_prev += pred.tolist()
            confs += conf.tolist()
    rel = classification_report(y_real, y_prev, target_names=config.CLASSES, digits=4)
    print(rel)
    (config.SAIDA / "relatorio_classificacao.txt").write_text(rel, encoding="utf-8")
    cm = confusion_matrix(y_real, y_prev)
    ConfusionMatrixDisplay(cm, display_labels=config.CLASSES).plot().figure_.savefig(config.SAIDA / "matriz_confusao.png", dpi=150)
    with open(config.SAIDA / "erros_teste.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["arquivo", "real", "previsto", "confianca"])
        for it, r, p, c in zip(ds.itens, y_real, y_prev, confs):
            if r != p:
                w.writerow([it["relativo"], config.CLASSES[r], config.CLASSES[p], round(c, 3)])
    print("Amostra de teste:", len(y_real), "| erros:", sum(r != p for r, p in zip(y_real, y_prev)))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/vc/codigo/analisar_erros.py
"""Fase 2: varredura de limiar no conjunto de TESTE e montagem visual dos erros.

Gera em evidencias/: varredura_limiar.csv e montagem_erros.png.
"""
import csv

import matplotlib
import torch
from PIL import Image
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.data import DataLoader

import config
from dados import Fissuras
from treinar import criar_modelo

matplotlib.use("Agg")
import matplotlib.pyplot as plt


def main():
    modelo = criar_modelo()
    modelo.load_state_dict(torch.load(config.MODELO, map_location="cpu"))
    modelo.eval()
    ds = Fissuras("teste")
    y, p = [], []
    with torch.no_grad():
        for x, r in DataLoader(ds, batch_size=config.BATCH):
            p += torch.softmax(modelo(x), 1)[:, 1].tolist()
            y += r.tolist()

    linhas = []
    for lim in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
        pred = [int(v >= lim) for v in p]
        fn = sum(1 for a, b in zip(y, pred) if a == 1 and b == 0)
        fp = sum(1 for a, b in zip(y, pred) if a == 0 and b == 1)
        linhas.append({"limiar": lim, "precisao": round(precision_score(y, pred), 4),
                       "recall": round(recall_score(y, pred), 4), "f1": round(f1_score(y, pred), 4),
                       "falsos_negativos": fn, "falsos_positivos": fp})
        print(linhas[-1])
    with open(config.SAIDA / "varredura_limiar.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(linhas[0]))
        w.writeheader()
        w.writerows(linhas)

    erros = [(it, r, v) for it, r, v in zip(ds.itens, y, p) if int(v >= config.LIMIAR_CONFIANCA) != r]
    cols = 4
    lin = (len(erros) + cols - 1) // cols
    fig, eixos = plt.subplots(lin, cols, figsize=(3.2 * cols, 3.4 * lin))
    for ax in eixos.flat:
        ax.axis("off")
    for ax, (it, r, v) in zip(eixos.flat, erros):
        ax.imshow(Image.open(it["arquivo"]).convert("RGB"))
        nome = it["arquivo"].replace("\\", "/").split("/")[-1]
        ax.set_title(f"{nome}\nreal={config.CLASSES[r]}\nP(fissura)={v:.2f}", fontsize=8)
    fig.tight_layout()
    fig.savefig(config.SAIDA / "montagem_erros.png", dpi=110)
    print("Erros:", len(erros))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/vc/codigo/exemplos_acertos.py
"""Monta exemplos de ACERTOS do conjunto de teste (limiar de config.LIMIAR_CONFIANCA).

Mostra, para cada classe, os acertos de maior confiança e os acertos mais "apertados"
(perto do limiar), para contrastar com erros no relatório. Saída: evidencias/montagem_acertos.png
"""
import matplotlib
import torch
from PIL import Image
from torch.utils.data import DataLoader

import config
from dados import Fissuras
from treinar import criar_modelo

matplotlib.use("Agg")
import matplotlib.pyplot as plt

N = 4


def main():
    modelo = criar_modelo()
    modelo.load_state_dict(torch.load(config.MODELO, map_location="cpu"))
    modelo.eval()
    ds = Fissuras("teste")
    p = []
    with torch.no_grad():
        for x, _ in DataLoader(ds, batch_size=config.BATCH):
            p += torch.softmax(modelo(x), 1)[:, 1].tolist()
    lim = config.LIMIAR_CONFIANCA
    acertos = [(it, int(it["rotulo"]), v) for it, v in zip(ds.itens, p) if int(v >= lim) == int(it["rotulo"])]
    grupos = []
    for rot in (1, 0):
        cl = [a for a in acertos if a[1] == rot]
        conf = lambda a: a[2] if rot == 1 else 1 - a[2]
        cl.sort(key=conf)
        grupos.append((f"{config.CLASSES[rot]}: mais confiantes", cl[::-1][:N]))
        grupos.append((f"{config.CLASSES[rot]}: acertos apertados", cl[:N]))
    fig, eixos = plt.subplots(len(grupos), N, figsize=(3.2 * N, 3.5 * len(grupos)))
    for lin, (titulo, itens) in enumerate(grupos):
        for col in range(N):
            ax = eixos[lin][col]
            ax.axis("off")
            if col < len(itens):
                it, r, v = itens[col]
                ax.imshow(Image.open(it["arquivo"]).convert("RGB"))
                ax.set_title(f"{titulo}\n{it['arquivo'].replace(chr(92), '/').split('/')[-1]}  P(fissura)={v:.2f}", fontsize=7)
    fig.tight_layout()
    fig.savefig(config.SAIDA / "montagem_acertos.png", dpi=110)
    print("acertos:", len(acertos), "limiar:", lim)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/vc/codigo/teste_robustez.py
"""Fase 2: teste de robustez no conjunto de TESTE com perturbações sintéticas.

Simula interferências citadas no enunciado (sombra, pouca luz, desfoque, obstáculo, compressão).
É um proxy: NÃO substitui fotos reais de fachadas e calçadas. Saída: evidencias/robustez.csv
"""
import csv
import io

import torch
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter
from sklearn.metrics import f1_score, precision_score, recall_score

import config
from dados import Fissuras, tf_base
from inferir import carregar_modelo


def sombra(img):
    w, h = img.size
    m = Image.new("L", (w, h), 255)
    ImageDraw.Draw(m).polygon([(0, 0), (w * 0.7, 0), (w * 0.3, h), (0, h)], fill=110)
    m = m.filter(ImageFilter.GaussianBlur(12))
    return Image.composite(img, Image.new("RGB", (w, h), (0, 0, 0)), m)


def obstaculo(img):
    w, h = img.size
    out = img.copy()
    ImageDraw.Draw(out).rectangle([w * 0.35, h * 0.35, w * 0.65, h * 0.65], fill=(60, 90, 40))
    return out


def jpeg(img):
    b = io.BytesIO()
    img.save(b, "JPEG", quality=15)
    return Image.open(b).convert("RGB")


PERTURBACOES = {
    "original": lambda i: i,
    "sombra": sombra,
    "pouca_luz": lambda i: ImageEnhance.Brightness(i).enhance(0.4),
    "superexposta": lambda i: ImageEnhance.Brightness(i).enhance(1.8),
    "desfoque": lambda i: i.filter(ImageFilter.GaussianBlur(3)),
    "obstaculo_central": obstaculo,
    "jpeg_q15": jpeg,
    "baixa_resolucao": lambda i: i.resize((48, 48)).resize((227, 227)),
}


def main():
    modelo = carregar_modelo()
    itens = Fissuras("teste").itens
    imgs = [(Image.open(it["arquivo"]).convert("RGB"), int(it["rotulo"])) for it in itens]
    linhas = []
    for nome, f in PERTURBACOES.items():
        y, p = [], []
        for img, r in imgs:
            with torch.no_grad():
                prob = torch.softmax(modelo(tf_base(f(img)).unsqueeze(0)), 1)[0, 1].item()
            y.append(r)
            p.append(int(prob >= config.LIMIAR_CONFIANCA))
        linhas.append({"perturbacao": nome, "precisao": round(precision_score(y, p), 4),
                       "recall": round(recall_score(y, p), 4), "f1": round(f1_score(y, p), 4),
                       "acuracia": round(sum(a == b for a, b in zip(y, p)) / len(y), 4)})
        print(linhas[-1])
    with open(config.SAIDA / "robustez.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(linhas[0]))
        w.writeheader()
        w.writerows(linhas)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/vc/codigo/exemplos_interferencias.py
"""Mostra, nas MESMAS imagens do teste, o efeito de sombra, pouca luz, superexposição, desfoque e obstáculo.

Imagens do dataset com interferência SIMULADA: não são fotos reais. Servem como análise visual de
como cada interferência muda a probabilidade de fissura. Amostra: 3 positivas e 3 negativas que o
modelo acertou sem interferência (sorteio com semente fixa, não escolhidas a dedo).
Saída: evidencias/montagem_interferencias.png e interferencias_exemplos.csv
"""
import csv
import random

import matplotlib
import torch
from PIL import Image

import config
from dados import Fissuras, tf_base
from inferir import carregar_modelo
from teste_robustez import PERTURBACOES

matplotlib.use("Agg")
import matplotlib.pyplot as plt

USAR = ["original", "sombra", "pouca_luz", "superexposta", "desfoque", "obstaculo_central"]


def main():
    modelo = carregar_modelo()
    lim = config.LIMIAR_CONFIANCA
    itens = Fissuras("teste").itens

    def prob(img):
        with torch.no_grad():
            return torch.softmax(modelo(tf_base(img).unsqueeze(0)), 1)[0, 1].item()

    rnd = random.Random(config.SEMENTE)
    escolhidos = []
    for rot in (1, 0):
        cand = []
        for it in itens:
            if int(it["rotulo"]) == rot:
                img = Image.open(it["arquivo"]).convert("RGB")
                if int(prob(img) >= lim) == rot:
                    cand.append(it)
        escolhidos += rnd.sample(cand, 3)

    fig, eixos = plt.subplots(len(escolhidos), len(USAR), figsize=(2.6 * len(USAR), 2.9 * len(escolhidos)))
    linhas = []
    for i, it in enumerate(escolhidos):
        base = Image.open(it["arquivo"]).convert("RGB")
        real = int(it["rotulo"])
        nome = it["arquivo"].replace("\\", "/").split("/")[-1]
        for j, pert in enumerate(USAR):
            img = PERTURBACOES[pert](base)
            p = prob(img)
            ok = int(p >= lim) == real
            ax = eixos[i][j]
            ax.imshow(img)
            ax.axis("off")
            ax.set_title(f"{pert}\nP(fissura)={p:.2f} {'OK' if ok else 'ERRO'}", fontsize=7, color="green" if ok else "red")
            linhas.append({"arquivo": nome, "real": config.CLASSES[real], "perturbacao": pert, "p_fissura": round(p, 4), "acerto": ok})
        eixos[i][0].text(-0.05, 0.5, f"{nome}\nreal={config.CLASSES[real]}", transform=eixos[i][0].transAxes,
                         ha="right", va="center", fontsize=7)
    fig.tight_layout()
    fig.savefig(config.SAIDA / "montagem_interferencias.png", dpi=110, bbox_inches="tight")
    with open(config.SAIDA / "interferencias_exemplos.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(linhas[0]))
        w.writeheader()
        w.writerows(linhas)
    erros = [l for l in linhas if not l["acerto"]]
    print(f"{len(linhas)} casos, {len(erros)} erros")
    for e in erros:
        print(e)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/vc/codigo/benchmark_video.py
"""Etapa 5: processa um vídeo curto (ou pasta de imagens) e mede latência média e FPS.

Uso: python benchmark_video.py caminho/video.mp4
     python benchmark_video.py pasta_com_imagens/
Registre no relatório: equipamento, resolução e resultados impressos.
"""
import platform
import statistics
import sys
import time
from pathlib import Path

import cv2
from PIL import Image

import config
from inferir import carregar_modelo, predizer


def quadros(origem: Path):
    if origem.is_dir():
        for p in sorted(origem.iterdir()):
            if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                yield Image.open(p)
        return
    cap = cv2.VideoCapture(str(origem))
    while True:
        ok, q = cap.read()
        if not ok:
            break
        yield Image.fromarray(cv2.cvtColor(q, cv2.COLOR_BGR2RGB))
    cap.release()


def main():
    origem = Path(sys.argv[1])
    carregar_modelo()  # aquecimento fora da medição
    tempos, alertas, res = [], 0, None
    t_total = time.perf_counter()
    for img in quadros(origem):
        res = res or img.size
        t0 = time.perf_counter()
        r = predizer(img)
        tempos.append(time.perf_counter() - t0)
        alertas += r["inspecao_recomendada"]
    total = time.perf_counter() - t_total
    if not tempos:
        raise SystemExit("Nenhum quadro processado. Verifique o caminho.")
    print(f"Equipamento: {platform.processor() or platform.machine()} | {platform.platform()}")
    print(f"Resolução dos quadros: {res[0]}x{res[1]} (redimensionado para {config.TAMANHO_IMG}x{config.TAMANHO_IMG})")
    print(f"Quadros: {len(tempos)} | alertas de inspeção: {alertas}")
    print(f"Latência média: {statistics.mean(tempos) * 1000:.1f} ms | mediana: {statistics.median(tempos) * 1000:.1f} ms")
    print(f"FPS (fim a fim): {len(tempos) / total:.1f}")


if __name__ == "__main__":
    main()

## 3. Dados: Concrete Crack Images for Classification (Mendeley)
Baixa o arquivo (241 MB), confere o hash e extrai em `imagens/Positive` e `imagens/Negative`.

In [ ]:
URL = 'https://data.mendeley.com/public-files/datasets/5y9wdsg2zt/files/8a70d8a5-bce9-4291-bab9-b48cfb3e87c3/file_downloaded'
SHA = '08d7dc505a4f5a0330cee2fa2a1ae4b5b4f98bcffd0549b774caf7959bb1f02f'
!mkdir -p /content/vc/imagens /content/dl
!curl -sL -o /content/dl/dataset.rar "$URL"
import hashlib
h = hashlib.sha256(open('/content/dl/dataset.rar','rb').read()).hexdigest()
assert h == SHA, 'hash diferente: download corrompido'
!bsdtar -xf /content/dl/dataset.rar -C /content/vc/imagens
!rm -rf /content/dl
!ls /content/vc/imagens/Positive | wc -l
!ls /content/vc/imagens/Negative | wc -l

## 4. Preparar: amostra e separação treino/validação/teste por blocos (sem vazamento)

In [ ]:
%cd /content/vc/codigo
!python preparar_dados.py

## 5. Modelo
**Opção A (padrão):** treinar aqui (MobileNetV2, 5 épocas; com GPU leva poucos minutos).
**Opção B:** enviar o `modelo_fissuras.pt` que você já treinou no computador, para repetir exatamente os números do relatório.
Mude `USAR_MODELO_LOCAL` para `True` se escolher a B.

In [ ]:
USAR_MODELO_LOCAL = False
if USAR_MODELO_LOCAL:
    from google.colab import files
    up = files.upload()                       # selecione modelo_fissuras.pt
    nome = next(iter(up))
    !mv "$nome" /content/vc/codigo/modelo_fissuras.pt
else:
    !python treinar.py

## 6. Limiar escolhido na VALIDAÇÃO e métricas no TESTE

In [ ]:
!python escolher_limiar.py | tail -n 25

In [ ]:
!python avaliar.py
from IPython.display import Image, display
display(Image('/content/vc/evidencias/matriz_confusao.png', width=420))

## 7. Análise de acertos e erros (conjunto de teste)

In [ ]:
!python analisar_erros.py | tail -n 9
!python exemplos_acertos.py
display(Image('/content/vc/evidencias/montagem_erros.png'))
display(Image('/content/vc/evidencias/montagem_acertos.png'))

## 8. Robustez com perturbações sintéticas (proxy; **não** substitui fotos reais)
Pode levar alguns minutos sem GPU.

In [ ]:
!python teste_robustez.py
import pandas as pd
display(pd.read_csv('/content/vc/evidencias/robustez.csv'))
!python exemplos_interferencias.py
display(Image('/content/vc/evidencias/montagem_interferencias.png'))

## 9. 📷 Suas fotos reais (requisito 3 do enunciado: sombra, textura, iluminação, obstáculos)

**Como fotografar (importante):** o modelo foi treinado com recortes de 227×227 px de superfícies **vistas de perto**.
Chegue a 30 a 60 cm da parede ou calçada, deixe a superfície ocupar o quadro todo e centralize a área de interesse.
Fotos de fachada inteira, de longe, fazem a fissura virar um traço de poucos pixels e o modelo erra por escala, não por defeito dele.

Faça **20 a 30 fotos**: cerca de metade com fissura e metade sem, e inclua algumas com `sombra`, `poucaluz`,
`obstaculo` (galho, objeto na frente), `textura` (reboco, pintura descascada, manchas) e `contraluz`.
Dê o nome com `com_` ou `sem_` no começo (veja a convenção no topo).

In [ ]:
from google.colab import files
up = files.upload()                           # selecione todas as fotos de uma vez
import shutil
for nome in up:
    shutil.move(nome, f'/content/vc/fotos_reais/{nome}')
print(len(up), 'fotos enviadas')

In [ ]:
import sys, re, csv
sys.path.insert(0, '/content/vc/codigo')
import config, inferir
from PIL import Image as PILImage, ImageOps
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

PASTA = BASE / 'fotos_reais'
EXT = {'.jpg', '.jpeg', '.png', '.webp'}

def quadrado(img):
    # recorte central quadrado (o modelo espera superfície vista de perto, 1:1)
    img = ImageOps.exif_transpose(img).convert('RGB')
    w, h = img.size
    l = min(w, h)
    return img.crop(((w - l) // 2, (h - l) // 2, (w - l) // 2 + l, (h - l) // 2 + l))

linhas = []
for p in sorted(PASTA.iterdir()):
    if p.suffix.lower() not in EXT:
        continue
    real = 1 if p.name.lower().startswith('com_') else 0 if p.name.lower().startswith('sem_') else None
    if real is None:
        print('IGNORADA (nome sem com_ ou sem_):', p.name)
        continue
    tags = re.findall(r'__([a-z]+)', p.stem.lower())
    r = inferir.predizer(quadrado(PILImage.open(p)))
    prev = 1 if r['inspecao_recomendada'] else 0
    linhas.append({'arquivo': p.name, 'real': config.CLASSES[real], 'previsto': config.CLASSES[prev],
                   'p_fissura': r['probabilidade_fissura'], 'confianca': r['confianca'],
                   'condicoes': ','.join(tags) or '-', 'acerto': real == prev, '_y': real, '_p': prev})
df = pd.DataFrame(linhas)
assert len(df), 'nenhuma foto válida: verifique os nomes (com_ ou sem_)'
df.drop(columns=['_y', '_p']).to_csv(BASE / 'evidencias' / 'fotos_reais_resultados.csv', index=False)
print(f'Limiar usado: {config.LIMIAR_CONFIANCA} | fotos: {len(df)} | acertos: {int(df.acerto.sum())} ({100*df.acerto.mean():.0f}%)')
if df._y.nunique() == 2:
    print(classification_report(df._y, df._p, target_names=config.CLASSES, digits=3, zero_division=0))
    ConfusionMatrixDisplay(confusion_matrix(df._y, df._p, labels=[0, 1]), display_labels=config.CLASSES).plot()
    plt.savefig(BASE / 'evidencias' / 'fotos_reais_matriz.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Envie fotos das duas classes (com_ e sem_) para calcular precisão, recall e F1.')
df.drop(columns=['_y', '_p'])

In [ ]:
# acerto por condição (sombra, poucaluz, obstaculo, textura...) e grade com as fotos
cond = df.assign(c=df.condicoes.str.split(',')).explode('c').groupby('c').agg(fotos=('acerto', 'size'), acertos=('acerto', 'sum'))
cond['taxa_acerto'] = (cond.acertos / cond.fotos).round(2)
display(cond)
cond.to_csv(BASE / 'evidencias' / 'fotos_reais_por_condicao.csv')

n = len(df); cols = 5; lin = (n + cols - 1) // cols
fig, eixos = plt.subplots(lin, cols, figsize=(3.2 * cols, 3.6 * lin))
for ax in eixos.flat: ax.axis('off')
for ax, (_, r) in zip(eixos.flat, df.iterrows()):
    ax.imshow(quadrado(PILImage.open(PASTA / r.arquivo)).resize((224, 224)))
    ax.set_title(f"{'ACERTO' if r.acerto else 'ERRO'} | real={r.real}\nP(fissura)={r.p_fissura:.2f}  {r.condicoes}", fontsize=7,
                 color='green' if r.acerto else 'red')
fig.tight_layout()
fig.savefig(BASE / 'evidencias' / 'fotos_reais_grade.png', dpi=110)
plt.show()

## 10. 🎥 Seu vídeo: latência média e FPS
Grave um vídeo **curto (10 a 20 s)** andando devagar ao longo de uma parede, muro ou calçada, com a câmera a 30 a 60 cm.
Anote a **resolução** do vídeo (a célula imprime) e o **equipamento** onde a medição foi feita.
O número que vale para o relatório é o do computador onde o app roda; repita `benchmark_video.py` lá
(`python benchmark_video.py caminho/do/video.mp4`). Aqui o Colab dá uma medição de referência em CPU.

In [ ]:
from google.colab import files
up = files.upload()                           # selecione o vídeo (MP4, MOV, AVI)
VIDEO = '/content/vc/' + next(iter(up))
import shutil; shutil.move(next(iter(up)), VIDEO)
print(VIDEO)

In [ ]:
%cd /content/vc/codigo
out = !python benchmark_video.py "$VIDEO"
print('\n'.join(out))
print(subprocess.run("lscpu | grep 'Model name'", shell=True, capture_output=True, text=True).stdout.strip())
open('/content/vc/evidencias/benchmark_video_real.txt', 'w').write('\n'.join(out))

In [ ]:
# probabilidade de fissura ao longo do vídeo (1 a cada 5 quadros)
import cv2, numpy as np
cap = cv2.VideoCapture(VIDEO); fps_v = cap.get(cv2.CAP_PROP_FPS) or 25; t, p, i = [], [], 0
while True:
    ok, q = cap.read()
    if not ok: break
    if i % 5 == 0:
        r = inferir.predizer(PILImage.fromarray(cv2.cvtColor(q, cv2.COLOR_BGR2RGB)))
        t.append(i / fps_v); p.append(r['probabilidade_fissura'])
    i += 1
cap.release()
plt.figure(figsize=(9, 3)); plt.plot(t, p); plt.axhline(config.LIMIAR_CONFIANCA, color='r', ls='--', label='limiar')
plt.xlabel('tempo (s)'); plt.ylabel('P(fissura)'); plt.legend(); plt.tight_layout()
plt.savefig('/content/vc/evidencias/video_real_probabilidade.png', dpi=130); plt.show()

## 11. Baixar todas as evidências geradas

In [ ]:
!cd /content/vc && zip -qr evidencias_colab.zip evidencias
from google.colab import files
files.download('/content/vc/evidencias_colab.zip')